# 04 — García Rodríguez multi-country EDA (D3, cross-market spine)

Supplement ec0010/mmc2 to *Automation in Construction* 133:104047 (CC BY-NC-ND 4.0). Unlike Mendeley, these market datasets include **losing bids** — full co-bid graphs and price screens apply (§4.3 D3).

In [1]:
import polars as pl

RESULTS = []

def check(name, expected, actual):
    """Soft verification: record PASS/FAIL, never crash mid-notebook."""
    ok = expected == actual
    RESULTS.append((name, expected, actual, ok))
    print(f"{'PASS' if ok else 'FAIL'}  {name}: expected={expected!r} actual={actual!r}")
    return ok

def summary():
    print("\n=== VERIFICATION SUMMARY ===")
    for name, exp, act, ok in RESULTS:
        print(f"{'PASS' if ok else 'FAIL'}  {name}: expected={exp!r} actual={act!r}")
    n_fail = sum(1 for r in RESULTS if not r[3])
    print(f"{len(RESULTS) - n_fail}/{len(RESULTS)} checks passed")


In [2]:
DATA = "../data/raw/garcia_rodriguez/extracted"
df = pl.read_csv(f"{DATA}/DB_Collusion_All_processed.csv",
                 infer_schema_length=20000, ignore_errors=True)
print(df.shape)
print(df.columns)
df.head(5)

(64348, 13)
['Tender', 'Bid_value', 'Winner', 'Number_bids', 'Collusive_competitor', 'CV', 'SPD', 'DIFFP', 'RD', 'KURT', 'SKEW', 'KSTEST', 'Dataset']


Tender,Bid_value,Winner,Number_bids,Collusive_competitor,CV,SPD,DIFFP,RD,KURT,SKEW,KSTEST,Dataset
i64,f64,f64,i64,f64,f64,f64,f64,f64,f64,f64,f64,i64
2,1.6171e7,1.0,11,0.0,0.2363,1.2746,0.0817,0.2978,0.7286,0.4923,0.2805,0
2,1.7493e7,0.0,11,0.0,0.2363,1.2746,0.0817,0.2978,0.7286,0.4923,0.2805,0
2,1.94925e7,0.0,11,0.0,0.2363,1.2746,0.0817,0.2978,0.7286,0.4923,0.2805,0
2,2.2554e7,0.0,11,0.0,0.2363,1.2746,0.0817,0.2978,0.7286,0.4923,0.2805,0
2,2.3675e7,0.0,11,0.0,0.2363,1.2746,0.0817,0.2978,0.7286,0.4923,0.2805,0


In [3]:
# Markets, bid-level structure, collusion labels
print(df.group_by("Dataset").agg(
    pl.len().alias("bid_rows"),
    pl.col("Tender").n_unique().alias("tenders"),
    pl.col("Collusive_competitor").mean().alias("collusive_share"),
    pl.col("Number_bids").mean().alias("avg_bids_per_tender"),
).sort("bid_rows", descending=True))
print(f"\ntotal bid rows: {df.height}, total tenders: {df['Tender'].n_unique()}")
print(f"losing bids present: {(df['Winner'] == 0).sum()} rows with Winner==0")

shape: (6, 5)
┌─────────┬──────────┬─────────┬─────────────────┬─────────────────────┐
│ Dataset ┆ bid_rows ┆ tenders ┆ collusive_share ┆ avg_bids_per_tender │
│ ---     ┆ ---      ┆ ---     ┆ ---             ┆ ---                 │
│ i64     ┆ u32      ┆ u32     ┆ f64             ┆ f64                 │
╞═════════╪══════════╪═════════╪═════════════════╪═════════════════════╡
│ 3       ┆ 21231    ┆ 4344    ┆ 0.802129        ┆ 6.109039            │
│ 1       ┆ 20286    ┆ 278     ┆ 0.423149        ┆ 91.850833           │
│ 5       ┆ 13515    ┆ 1080    ┆ 0.11269         ┆ 13.276138           │
│ 2       ┆ 7004     ┆ 3754    ┆ 0.177756        ┆ 2.338521            │
│ 4       ┆ 1629     ┆ 224     ┆ 0.81768         ┆ 8.11725             │
│ 0       ┆ 683      ┆ 101     ┆ 0.20937         ┆ 9.330893            │
└─────────┴──────────┴─────────┴─────────────────┴─────────────────────┘

total bid rows: 64348, total tenders: 9781
losing bids present: 54389 rows with Winner==0


In [4]:
# Screens shipped with the data (fused as features per §4.4)
screens = ["CV", "SPD", "DIFFP", "RD", "KURT", "SKEW", "KSTEST"]
print(df.select([pl.col(s).is_not_null().mean().alias(s) for s in screens]))
summary()

shape: (1, 7)
┌─────┬─────┬───────┬─────┬──────┬──────┬────────┐
│ CV  ┆ SPD ┆ DIFFP ┆ RD  ┆ KURT ┆ SKEW ┆ KSTEST │
│ --- ┆ --- ┆ ---   ┆ --- ┆ ---  ┆ ---  ┆ ---    │
│ f64 ┆ f64 ┆ f64   ┆ f64 ┆ f64  ┆ f64  ┆ f64    │
╞═════╪═════╪═══════╪═════╪══════╪══════╪════════╡
│ 1.0 ┆ 1.0 ┆ 1.0   ┆ 1.0 ┆ 1.0  ┆ 1.0  ┆ 1.0    │
└─────┴─────┴───────┴─────┴──────┴──────┴────────┘

=== VERIFICATION SUMMARY ===
0/0 checks passed


**Conclusions.** Bid-level data with losing bids across all markets — the co-bid graph tier and bid-price screens are fully supported here; LOMO/LOCO splits per §4.3 D3 are feasible. Counts feed the Week-2 procurement adapter.